# Notebook 09b — Fine-tuning control (baseline + 3 epochs)

**Objetivo:** verificar si la mejora observada en NB09 (modelo comprimido + 3 epochs FT supera al baseline) viene del efecto regularizador de la SVD o simplemente del entrenamiento adicional.

**Diseño:** tomar el baseline `bert-goemotions-23emo-final` (NO comprimido) y aplicar 3 épocas adicionales de fine-tuning con la MISMA configuración que NB09 usó sobre el modelo comprimido.

**Decisión:**
- Si F1 baseline+3ep ≈ F1 baseline original (0,577): la mejora del comprimido+FT viene de la regularización por SVD. **Hallazgo confirmado.**
- Si F1 baseline+3ep ≈ F1 comprimido+FT (0,591): la mejora viene del entrenamiento adicional, no de la compresión. **Hay que rebajar el claim.**
- Caso intermedio: ambos efectos contribuyen, hay que cuantificar.

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    !pip install -q transformers datasets scikit-learn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments

from src.data import load_goemotions
from src.models import load_bert_classifier
from src.utils import compute_metrics

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

results_dir = os.path.join(PROJECT_ROOT, 'results')
out_dir = os.path.join(results_dir, 'notebook9b')
os.makedirs(out_dir, exist_ok=True)
print(f'Output directory: {out_dir}')

In [ ]:
# Cargar datos y baseline (idéntico a NB09)
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions(exclude_emotions=EXCLUDED_EMOTIONS)

MODEL_PATH = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-23emo-final')
baseline_model = load_bert_classifier(model_name=MODEL_PATH, num_labels=len(emotion_names))
baseline_model.to(device)
baseline_model.eval()

num_labels = len(emotion_names)
print(f'Baseline cargado desde {MODEL_PATH}')
print(f'num_labels: {num_labels}, test set: {len(test_ds)} samples')

## Evaluación pre-FT (debería dar F1 ≈ 0,577)

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataset, data_collator, device, batch_size=64):
    """Evalúa: macro F1, micro F1, F1 por emoción. Idéntico a NB09."""
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator, shuffle=False)
    all_preds, all_labels = [], []

    model.eval()
    for batch in tqdm(loader, desc='Evaluating', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits) > 0.5).float()
        all_preds.append(preds)
        all_labels.append(labels)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    per_emotion = [f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
                   for i in range(all_labels.shape[1])]
    return {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'per_emotion_f1': np.array(per_emotion),
    }

print('Evaluando baseline (pre-FT)...')
baseline_pre = evaluate_model(baseline_model, test_ds, data_collator, device)
print(f'\nBaseline original:')
print(f'  macro_f1 = {baseline_pre["macro_f1"]:.4f}  (esperado ~0,577)')
print(f'  micro_f1 = {baseline_pre["micro_f1"]:.4f}')

## Fine-tuning de control: 3 épocas adicionales sobre el baseline

**Misma configuración** que NB09 usó para el modelo comprimido (greedy_90):
- 3 épocas
- learning rate = 2e-5
- warmup ratio = 0,10
- batch size = 32
- BCE multi-label loss (la del propio `BertForSequenceClassification` con `problem_type=multi_label_classification`)
- semilla 42

In [ ]:
def finetune_baseline(model, train_ds, val_ds, data_collator,
                      epochs=3, lr=2e-5, batch_size=32, output_dir=None, seed=42):
    """Fine-tune el baseline sin compresión (control)."""
    if output_dir is None:
        output_dir = os.path.join(PROJECT_ROOT, 'results', 'finetuned_baseline_control_tmp')

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=100,
        report_to='none',
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        seed=seed,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )
    trainer.train()
    return model

print('Iniciando fine-tuning del baseline (3 epochs)...')
baseline_model = finetune_baseline(
    baseline_model, train_ds, val_ds, data_collator,
    epochs=3, lr=2e-5,
)
baseline_model.to(device)

## Evaluación post-FT

In [ ]:
print('Evaluando baseline post-FT...')
baseline_post = evaluate_model(baseline_model, test_ds, data_collator, device)

delta = baseline_post['macro_f1'] - baseline_pre['macro_f1']
print(f'\n=== RESULTADO DEL EXPERIMENTO DE CONTROL ===')
print(f'  F1 macro baseline original     : {baseline_pre["macro_f1"]:.4f}')
print(f'  F1 macro baseline + 3 epochs FT: {baseline_post["macro_f1"]:.4f}')
print(f'  Delta                          : {delta:+.4f}')
print()
print(f'  (Recordatorio NB09: F1 comprimido+FT = 0,591)')
print()
if delta < 0.005:
    print('  >> Mejora del comprimido+FT NO viene del entrenamiento adicional.')
    print('  >> Hipótesis de regularización implícita por SVD: APOYADA.')
elif delta > 0.014:
    print('  >> Mejora del comprimido+FT viene del entrenamiento adicional.')
    print('  >> Hipótesis de regularización implícita por SVD: REFUTADA.')
else:
    print('  >> Caso intermedio: parte del efecto es regularización, parte entrenamiento.')
    print('  >> Habrá que cuantificar la contribución relativa.')

## Análisis por emoción

Importante: comparar qué emociones ganan/pierden con FT adicional sobre el baseline.
Si la mejora del comprimido se concentra en emociones infrarrepresentadas (embarrassment, etc.),
y el control también las mejora con magnitud similar, el efecto es del FT, no de la SVD.

In [ ]:
# Cargar resultado del comprimido+FT desde NB09 para comparar las tres condiciones
compressed_ft_path = os.path.join(results_dir, 'notebook9', 'finetuning_recovery.csv')
comp_ft_df = pd.read_csv(compressed_ft_path)

control_df = pd.DataFrame({
    'emotion': emotion_names,
    'baseline_f1': baseline_pre['per_emotion_f1'],
    'baseline_3ep_f1': baseline_post['per_emotion_f1'],
    'compressed_3ep_f1': comp_ft_df['finetuned_f1'].values,
})
control_df['baseline_3ep_delta'] = control_df['baseline_3ep_f1'] - control_df['baseline_f1']
control_df['compressed_delta']   = control_df['compressed_3ep_f1'] - control_df['baseline_f1']
control_df['svd_specific_gain']  = control_df['compressed_3ep_f1'] - control_df['baseline_3ep_f1']

print('\nComparación por emoción (las tres condiciones):')
print(control_df.to_string(index=False, float_format=lambda x: f'{x:+.4f}' if x != int(x) else f'{x:.0f}'))

In [ ]:
# Visualización: tres barras por emoción
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

order = np.argsort(control_df['baseline_f1'].values)[::-1]
emo_sorted = control_df['emotion'].values[order]
x = np.arange(len(emo_sorted))
w = 0.27

ax = axes[0]
ax.bar(x - w, control_df['baseline_f1'].values[order], w,
       label='Baseline original', color='gray', alpha=0.85)
ax.bar(x,     control_df['baseline_3ep_f1'].values[order], w,
       label='Baseline + 3ep FT (control)', color='steelblue', alpha=0.85)
ax.bar(x + w, control_df['compressed_3ep_f1'].values[order], w,
       label='Greedy_90 + 3ep FT', color='crimson', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(emo_sorted, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1 por emoción')
ax.set_title('A) F1 por emoción: tres condiciones')
ax.legend()

ax = axes[1]
delta_order = np.argsort(control_df['svd_specific_gain'].values)
emo_sorted2 = control_df['emotion'].values[delta_order]
ax.barh(np.arange(len(emo_sorted2)), control_df['svd_specific_gain'].values[delta_order],
        color=['green' if v > 0 else 'red' for v in control_df['svd_specific_gain'].values[delta_order]],
        alpha=0.8)
ax.set_yticks(np.arange(len(emo_sorted2))); ax.set_yticklabels(emo_sorted2, fontsize=8)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('SVD-specific gain = F1(compressed+FT) - F1(baseline+FT)')
ax.set_title('B) Mejora atribuible específicamente a la compresión')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'control_three_conditions.png'), dpi=150, bbox_inches='tight')
plt.show()

## Export

In [ ]:
control_df.to_csv(os.path.join(out_dir, 'control_per_emotion.csv'), index=False)

summary = pd.DataFrame([{
    'condition': 'baseline_original',
    'macro_f1': baseline_pre['macro_f1'],
    'micro_f1': baseline_pre['micro_f1'],
    'params_ratio': 1.0,
}, {
    'condition': 'baseline_plus_3ep_FT',
    'macro_f1': baseline_post['macro_f1'],
    'micro_f1': baseline_post['micro_f1'],
    'params_ratio': 1.0,
}, {
    'condition': 'greedy90_plus_3ep_FT (from NB09)',
    'macro_f1': comp_ft_df['finetuned_f1'].mean(),
    'micro_f1': float('nan'),
    'params_ratio': 0.864,
}])
summary.to_csv(os.path.join(out_dir, 'control_summary.csv'), index=False)
print(summary.to_string(index=False))
print(f'\nResultados guardados en: {out_dir}')